# Target variables
This notebook contains all code which was used for calculating the target variables, inclunding:

- A(UHII): Amplification of UHII during heatwave days
- $\Delta$UHII: Scenario Difference of UHII (thermodynamic climate change signal)
- A($\Delta$UHII): Amplification of $\Delta$UHII during heatwave days

The target variable UHII was already calculated in the 00_Data_Pre-Processing.ipynb notebook. 

For further analysis the <b>summer mean</b> of each target variable is calculated. 

# Load packages

In [9]:
import numpy as np
import pandas as pd 
import xarray as xr
from pathlib import Path
import os
import warnings


from scipy.stats import linregress
from scipy import stats as scipy_stats
from collections import Counter

# Set working directories

In [2]:
save_script_to = "/home/b/b383801/"
save_data_to = "/work/bb1445/b383801/"
get_data_from = "/work/bb1152/m300755/inputdata/IFS-FESOM/"
save_tables_to = "/home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/"
save_figures_to = "/home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Figures/"

# Functions

## Calculate summer metrics (mean summer temperature, per heatwave percentile, mean diurnal temperature (Tmax - Tmin), per heatwave percentile, mean summer UHII, per heatwave percentile)

<b> Calculates</b>:

    1. Summer mean temperatures (T2M_mean_mean_[cell]_summer)
    2. Heatwave-specific temperatures (T2M_mean_mean_[cell]_summer_[percentile])
    3. Summer mean UHII (UHII_summer)
    4. Heatwave-specific UHII (UHII_summer_[percentile])
    5. Summer mean diurnal range (Diurnal_Tmax_Tmin_diff_[cell]_summer)
    6. Heatwave diurnal range (Diurnal_Tmax_Tmin_diff_[cell]_summer_[percentile])
    
IMPORTANT: Heatwave days are ALWAYS defined by HIST dataset, even when processing CONT or T2K scenarios. This ensures consistent comparison.

### Only single scenario
`_process_single_scenario` function

In [3]:
def _process_single_scenario(ds_scenario, 
                             ds_hist_for_mask, 
                             cell_types, 
                             heatwave_percentiles, 
                             scenario_name):
    
    print(f"\nProcessing {scenario_name}...")
    
    scenario_results = []
    
    for city in ds_scenario.city.values:
        city_data = ds_scenario.sel(city=city)
        city_hist_data = ds_hist_for_mask.sel(city=city)
        
        # Determine hemisphere
        lat = float(ds_scenario.lat.sel(city=city).values)
        summer_months = [6, 7, 8] if lat >= 0 else [12, 1, 2]

        common_times = np.intersect1d(city_data.time.values, city_hist_data.time.values)
        if len(common_times) == 0:
            print(f"Warning: No common time overlap for city {city} between {scenario_name} and HIST")
            continue

        city_summer_aligned = city_data.sel(time=common_times)     
        city_hist_summer = city_hist_data.sel(time=common_times)

        hist_summer_mask = city_hist_summer.time.dt.month.isin(summer_months)
        city_hist_summer = city_hist_summer.sel(time=hist_summer_mask)

        city_summer_final = city_summer_aligned.sel(time=hist_summer_mask)
        
        city_metrics = {}

        # All summer days   
        for cell in cell_types:
            temp_var = f"T2M_mean_mean_{cell}"
            if temp_var in city_summer_final.data_vars:
                city_metrics[f"{temp_var}_summer"] = city_summer_final[temp_var].mean(dim='time')

        if "UHII" in city_summer_final.data_vars:
            city_metrics["UHII_summer"] = city_summer_final["UHII"].mean(dim='time')

        # Calculate diurnal range (Tmax - Tmin)
        for cell in cell_types:
            max_var = f"T2M_max_mean_{cell}"
            min_var = f"T2M_min_mean_{cell}"
            if max_var in city_summer_final and min_var in city_summer_final:
                diurnal_range = city_summer_final[max_var] - city_summer_final[min_var]
                city_metrics[f"Diurnal_Tmax_Tmin_diff_{cell}_summer"] = diurnal_range.mean(dim='time')

        # No-hw mask (days where all percentile flags are 0)
        no_hw_mask = xr.ones_like(city_hist_summer.time, dtype=bool) 
        no_hw_mask_scenario = xr.ones_like(city_summer_final.time, dtype=bool)
        
        for percentile in heatwave_percentiles:
            hw_flag_var = f"T2M_mean_mean_urban_high_extreme_{percentile}_heatwave"
            if hw_flag_var in city_hist_summer.data_vars:
                no_hw_mask = no_hw_mask & (city_hist_summer[hw_flag_var] == 0)

            if hw_flag_var in city_summer_final.data_vars:
                no_hw_mask_scenario = no_hw_mask_scenario & (city_summer_final[hw_flag_var] == 0)
        
        city_summer_no_hw = city_summer_final.where(no_hw_mask, drop = False)
        city_metrics["n_days_summer_no_hw"] = int(no_hw_mask_scenario.sum().values) 

        for cell in cell_types:
            temp_var = f"T2M_mean_mean_{cell}"
            if temp_var in city_summer_no_hw.data_vars:
                city_metrics[f"{temp_var}_summer_no_hw"] = \
                    city_summer_no_hw[temp_var].mean(dim='time', skipna=True)

        if "UHII" in city_summer_no_hw.data_vars:
            city_metrics["UHII_summer_no_hw"] = \
                city_summer_no_hw["UHII"].mean(dim='time', skipna=True)

        for cell in cell_types:
            max_var = f"T2M_max_mean_{cell}"
            min_var = f"T2M_min_mean_{cell}"
            if max_var in city_summer_no_hw and min_var in city_summer_no_hw:
                diurnal_no_hw = city_summer_no_hw[max_var] - city_summer_no_hw[min_var]
                city_metrics[f"Diurnal_Tmax_Tmin_diff_{cell}_summer_no_hw"] = \
                    diurnal_no_hw.mean(dim='time', skipna=True)           
        
        # Heatwave-specific metrics - Heatwave flag from HIST (always)
        for percentile in heatwave_percentiles:
            hw_flag_var = f"T2M_mean_mean_urban_high_extreme_{percentile}_heatwave"
            
            if hw_flag_var not in city_hist_summer.data_vars:
                continue
                
            hw_mask = city_hist_summer[hw_flag_var] == 1
            hw_mask_scenario = city_summer_final[hw_flag_var] == 1
            city_summer_hw = city_summer_final.where(hw_mask, drop=False)

            # Count heatwave days
            city_metrics[f"n_hwd_summer_{percentile}"] = int(hw_mask_scenario.sum().values)
                
            # Temperature means during heatwaves
            for cell in cell_types:
                temp_var = f"T2M_mean_mean_{cell}"
                if temp_var in city_summer_hw.data_vars:
                    city_metrics[f"{temp_var}_summer_{percentile}"] = \
                        city_summer_hw[temp_var].mean(dim='time', skipna=True)
                        
            # UHII during heatwaves
            if "UHII" in city_summer_hw.data_vars:
                city_metrics[f"UHII_summer_{percentile}"] = \
                    city_summer_hw["UHII"].mean(dim='time', skipna=True)
                
            # Diurnal range during heatwaves
            for cell in cell_types:
                max_var = f"T2M_max_mean_{cell}"
                min_var = f"T2M_min_mean_{cell}"
                
                if max_var in city_summer_hw and min_var in city_summer_hw:
                    diurnal_hw = city_summer_hw[max_var] - city_summer_hw[min_var]
                    city_metrics[f"Diurnal_Tmax_Tmin_diff_{cell}_summer_{percentile}"] = \
                        diurnal_hw.mean(dim='time', skipna=True)
        
        if city_metrics:
            city_result = xr.Dataset(city_metrics, coords={'city': city})
            scenario_results.append(city_result)
    
    # Combine all cities
    ds_result = xr.concat(scenario_results, dim='city')
    
    # Add coordinates from original dataset
    for coord in ["kg_class_main", "kg_class_detailed", "kg_diversity",
                  "kg_confidence_city", "lat", "lon"]:
        if coord in ds_scenario.coords:
            ds_result.coords[coord] = ds_scenario.coords[coord]
    
    # Add metadata
    ds_result.attrs = {
        "description": f"Summer climate metrics for {scenario_name} scenario",
        "summer_definition": "NH: JJA (Jun-Aug), SH: DJF (Dec-Feb)",
        "scenario": scenario_name,
        "heatwave_reference": "HIST" if scenario_name != "HIST" else "self",
        "note": "Heatwave days defined by HIST dataset for all scenarios",
    }
    
    return ds_result

### Absolute Scenario difference
`_calculate_scenario_diff` function

In [29]:
def _calculate_abs_scenario_diff(ds1, 
                                 ds2, 
                                 name1, 
                                 name2,
                                 substract_order="first_minus_second"): # "first_minus_second" or "second_minus_first"
    common_vars = set(ds1.data_vars).intersection(set(ds2.data_vars))
    
    ds_diff = xr.Dataset(coords=ds1.coords)

    if substract_order == "first_minus_second":
        left, right = ds1, ds2
        left_name, right_name = name1, name2

    elif substract_order == "second_minus_first":
        left, right = ds2, ds1
        left_name, right_name = name2, name1

    else:
        raise ValueError("substract_order must be 'first_minus_second' or 'second_minus_first'")
    
    for var in common_vars:
        out_name = f"{var}_{left_name}_{right_name}_abs_diff"
               
        diff = abs(left[var]) - abs(right[var])
       
        diff.attrs = {
            "long_name": f"Absolute scenario difference for {var}",
            "description": f"{left_name} minus {right_name}: {var}",
            "units": ds1[var].attrs.get("units", "K"),
            "formula": f"{left_name} - {right_name}",
        }
        
        ds_diff[out_name] = diff
    
    ds_diff.attrs = {
        "description": f"Climate scenario absolute differences ({left_name} - {right_name})",
        "scenario_calculation": f"{left_name} - {right_name}",
        "n_variables": len(common_vars),
        "summer_definition": ds1.attrs.get("summer_definition", "NH: JJA, SH: DJF"),
    }
    
    return ds_diff

### Creating final dataset with all new variables
`calculate_summer_climate_metrics` function

In [5]:
def calculate_summer_climate_metrics(ds_hist,
                                     ds_cont=None,
                                     ds_t2k=None,
                                     heatwave_percentiles=["p90", "p95", "p98"],
                                     diff="abs",
                                     save_path=None,
                                     output_name=None):
    
    valid_percentiles = ["p90", "p95", "p98"]
    for p in heatwave_percentiles:
        if p not in valid_percentiles:
            raise ValueError(f"Invalid percentile: {p}. Must be one of {valid_percentiles}")
    
    cell_types = ['urban_high', 'urban_low', 'rural']
    
    print("Calculating summer climate metrics per city...") 

    print("\n Processing HIST scenario")
    ds_hist_metrics = _process_single_scenario(ds_hist, 
                                               ds_hist,  # Use HIST for heatwave mask
                                               cell_types,
                                               heatwave_percentiles,
                                               scenario_name="HIST")

    ds_cont_metrics = None
    ds_t2k_metrics = None
    ds_hist_cont_diff = None
    ds_hist_t2k_diff = None
    
    if ds_cont is not None:
        print("\n Processing CONT scenario")
        print("Using HIST heatwave days for masking")
        ds_cont_metrics = _process_single_scenario(ds_cont,
                                                   ds_hist,  # Use HIST for heatwave mask
                                                   cell_types,
                                                   heatwave_percentiles,
                                                   scenario_name="CONT")

        if diff == "abs":
        # absolute
            print("\n Calculating HIST - CONT absolute differences")
            ds_hist_cont_diff = _calculate_abs_scenario_diff(ds_hist_metrics,
                                                             ds_cont_metrics,
                                                             "HIST",
                                                             "CONT",
                                                             substract_order="first_minus_second")
        else:
            # relative
            print("\n Calculating HIST - CONT relative differences")
            ds_hist_cont_diff = _calculate_rel_scenario_diff(ds_hist_metrics,
                                                             ds_cont_metrics,
                                                             "HIST",
                                                             "CONT",
                                                             substract_order="first_minus_second")

    if ds_t2k is not None:
        print("\n Processing T2K scenario")
        print("Using HIST heatwave days for masking")
        ds_t2k_metrics = _process_single_scenario(ds_t2k,
                                                  ds_hist,  # Use HIST for heatwave mask
                                                  cell_types,
                                                  heatwave_percentiles,
                                                  scenario_name="T2K")
        
        if diff == "abs":
            # absolute
            print("\n Calculating HIST - T2K absolute differences")
            ds_hist_t2k_diff = _calculate_abs_scenario_diff(ds_hist_metrics,
                                                            ds_t2k_metrics,
                                                            "HIST",
                                                            "T2K",
                                                            substract_order="second_minus_first")
        else:
            # relative
            print("\n Calculating HIST - T2K realtive differences")
            ds_hist_t2k_diff = _calculate_rel_scenario_diff(ds_hist_metrics,
                                                            ds_t2k_metrics,
                                                            "HIST",
                                                            "T2K",
                                                            substract_order="second_minus_first")
    
    if save_path is not None and output_name is not None:     
    
        filename = output_name.replace(".nc", "_HIST.nc")
        full_path = os.path.join(save_path, filename)
        ds_hist_metrics.to_netcdf(full_path)

        if ds_cont_metrics is not None:
            filename = output_name.replace(".nc", "_CONT.nc")
            full_path = os.path.join(save_path, filename)
            ds_cont_metrics.to_netcdf(full_path)
        
        if ds_t2k_metrics is not None:
            filename = output_name.replace(".nc", "_T2K.nc")
            full_path = os.path.join(save_path, filename)
            ds_t2k_metrics.to_netcdf(full_path)
    
        if ds_hist_cont_diff is not None:
            filename = output_name.replace(".nc", "_HIST_CONT_diff.nc")
            full_path = os.path.join(save_path, filename)
            ds_hist_cont_diff.to_netcdf(full_path)
        
        if ds_hist_t2k_diff is not None:
            filename = output_name.replace(".nc", "_HIST_T2K_diff.nc")
            full_path = os.path.join(save_path, filename)
            ds_hist_t2k_diff.to_netcdf(full_path) 
        
    return ds_hist_metrics, ds_cont_metrics, ds_t2k_metrics, ds_hist_cont_diff, ds_hist_t2k_diff

## Create data table with metrics per heatwave condition and climate class

### UHII

In [7]:
def create_uhii_summary_table(ds, 
                              save_path=None, 
                              output_name=None):

    conditions = {"HIST All Summer Days": "UHII_summer",
                  "HIST No-HW Summer Days": "UHII_summer_no_hw",
                  "HIST p95-HW Summer Days": "UHII_summer_p95"}

    kg_groups = {"All cities": None,
                 "Tropical": "A",
                 "Arid": "B",
                 "Temperate": "C",
                 "Continental": "D"}

    kg_values = ds["kg_class_main"].values

    metrics = ["Data range min [K]",
               "Data range max [K]",
               "Q1 - 1.5*IQR [K]",
               "Q3 + 1.5*IQR [K]",
               "Median UHII [K]",
               "N outliers (> 1.5*IQR)",
               "N cities UHII > 0",
               "N cities UHII < 0",
               "N cities total"]

    multi_cols = pd.MultiIndex.from_product([list(conditions.keys()), list(kg_groups.keys())],
                                            names=["Scenario / Condition", "Climate Group"])

    df_out = pd.DataFrame(index=metrics, columns=multi_cols)
    df_out.index.name = "Metric"

    for cond_label, var_name in conditions.items():
        if var_name not in ds.data_vars:
            print(f"Warning: '{var_name}' not found, skipping")
            continue

        all_vals = ds[var_name].values

        for group_label, kg_filter in kg_groups.items():

            vals = all_vals if kg_filter is None else all_vals[kg_values == kg_filter]
            vals = vals[~np.isnan(vals)]

            q1, q3 = np.nanpercentile(vals, [25, 75])
            iqr    = q3 - q1
            low_iqr = q1 - 1.5 * iqr
            high_iqr = q3 + 1.5 * iqr
            n_out  = int(np.sum((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)))

            df_out.loc["Data range min [K]", (cond_label, group_label)] = f"{np.min(vals):.3f}"
            df_out.loc["Data range max [K]", (cond_label, group_label)] = f"{np.max(vals):.3f}"
            df_out.loc["Q1 - 1.5*IQR [K]", (cond_label, group_label)] = f"{low_iqr:.3f}"
            df_out.loc["Q3 + 1.5*IQR [K]", (cond_label, group_label)] = f"{high_iqr:.3f}"
            df_out.loc["Median UHII [K]",  (cond_label, group_label)] = f"{np.median(vals):.3f}"
            df_out.loc["N outliers (> 1.5*IQR)", (cond_label, group_label)] = n_out
            df_out.loc["N cities UHII > 0", (cond_label, group_label)] = int(np.sum(vals > 0))
            df_out.loc["N cities UHII < 0", (cond_label, group_label)] = int(np.sum(vals < 0))
            df_out.loc["N cities total", (cond_label, group_label)] = len(vals)

    if save_path is not None:
        import os
        df_out.to_csv(os.path.join(save_path, output_name))
        print(f"Saved to {os.path.join(save_path, output_name)}")

    return df_out

### $\Delta$ UHII

In [8]:
def create_delta_uhii_summary_table(ds, 
                              save_path=None, 
                              output_name=None):

    conditions = {"HIST-CONT All Summer Days": "UHII_summer_HIST_CONT_abs_diff",
                  "HIST-CONT No-HW Summer Days": "UHII_summer_no_hw_HIST_CONT_abs_diff",
                  "HIST-CONT p95-HW Summer Days": "UHII_summer_p95_HIST_CONT_abs_diff"}

    kg_groups = {"All cities": None,
                 "Tropical": "A",
                 "Arid": "B",
                 "Temperate": "C",
                 "Continental": "D"}

    kg_values = ds["kg_class_main"].values

    metrics = ["Data range min [K]",
               "Data range max [K]",
               "Q1 - 1.5*IQR [K]",
               "Q3 + 1.5*IQR [K]",
               "Median $\\Delta$UHII [K]",
               "N outliers (> 1.5*IQR)",
               "N cities $\\Delta$UHII > 0",
               "N cities $\\Delta$UHII < 0",
               "N cities total"]

    multi_cols = pd.MultiIndex.from_product([list(conditions.keys()), list(kg_groups.keys())],
                                            names=["Scenario / Condition", "Climate Group"])

    df_out = pd.DataFrame(index=metrics, columns=multi_cols)
    df_out.index.name = "Metric"

    for cond_label, var_name in conditions.items():
        if var_name not in ds.data_vars:
            print(f"Warning: '{var_name}' not found, skipping")
            continue

        all_vals = ds[var_name].values

        for group_label, kg_filter in kg_groups.items():

            vals = all_vals if kg_filter is None else all_vals[kg_values == kg_filter]
            vals = vals[~np.isnan(vals)]

            q1, q3 = np.nanpercentile(vals, [25, 75])
            iqr    = q3 - q1
            low_iqr = q1 - 1.5 * iqr
            high_iqr = q3 + 1.5 * iqr
            n_out  = int(np.sum((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)))

            df_out.loc["Data range min [K]", (cond_label, group_label)] = f"{np.min(vals):.3f}"
            df_out.loc["Data range max [K]", (cond_label, group_label)] = f"{np.max(vals):.3f}"
            df_out.loc["Q1 - 1.5*IQR [K]", (cond_label, group_label)] = f"{low_iqr:.3f}"
            df_out.loc["Q3 + 1.5*IQR [K]", (cond_label, group_label)] = f"{high_iqr:.3f}"
            df_out.loc["Median $\\Delta$UHII [K]", (cond_label, group_label)] = f"{np.median(vals):.3f}"
            df_out.loc["N outliers (> 1.5*IQR)", (cond_label, group_label)] = n_out
            df_out.loc["N cities $\\Delta$UHII > 0", (cond_label, group_label)] = int(np.sum(vals > 0))
            df_out.loc["N cities $\\Delta$UHII < 0", (cond_label, group_label)] = int(np.sum(vals < 0))
            df_out.loc["N cities total", (cond_label, group_label)] = len(vals)

    if save_path is not None:
        import os
        df_out.to_csv(os.path.join(save_path, output_name))
        print(f"Saved to {os.path.join(save_path, output_name)}")

    return df_out

## Calculate amplification index <i>A</i>
For `T2M_mean_mean_urban_high_summer`, `T2M_mean_mean_rural_summer`, `UHII_summer` between no-heatwave days and p95 heatwave days.

In [3]:
def calculate_amplification_index(ds,
                                 heatwave_condition="p95",
                                 variables=["T2M_mean_mean_urban_high_summer", "T2M_mean_mean_rural_summer", "UHII_summer"],
                                 suffix=None,
                                 save_path=None,
                                 output_name=None):
    ds_out = ds.copy()

    for var in variables:
        if suffix is None:
            var_no_hw = f"{var}_no_hw"
            var_hw = f"{var}_{heatwave_condition}"
        else:
            var_no_hw = f"{var}_no_hw_{suffix}"
            var_hw = f"{var}_{heatwave_condition}_{suffix}"
        
        if var_hw not in ds.data_vars or var_no_hw not in ds.data_vars:
            print(f"WARNING: skipping '{var}' — could not find '{var_hw}' or '{var_no_hw}' in dataset")
            continue

        amplification = abs(ds[var_hw]) - abs(ds[var_no_hw])
        
        amp_name = f"amplification_{var}"
        amplification.name = amp_name
        amplification.attrs = {"long_name": f"{var} amplification",
                               "units": ds[var_hw].attrs.get("units", "K"),
                               "description": f"Amplification between heatwave days ({heatwave_condition}) and non-heatwave days",
                               "calculation": f"A = {var}_{heatwave_condition} - {var}_no_hw"}

        ds_out[amp_name] = amplification
    
    if save_path is not None and output_name is not None:
        full_path = os.path.join(save_path, output_name)
        ds_out.to_netcdf(full_path)

    return ds_out    

## Create data table with A(UHII) per climate class

In [10]:
def create_a_uhii_summary_table(ds, 
                                save_path=None, 
                                output_name=None):

    conditions = {"HIST - CONT": "amplification_UHII_summer"}

    kg_groups = {"All cities": None,
                 "Tropical":  "A",
                 "Arid":  "B",
                 "Temperate":  "C",
                 "Continental":  "D"}

    kg_values = ds["kg_class_main"].values

    metrics = ["Data range min [K]",
               "Data range max [K]",
               "Q1 - 1.5*IQR [K]",
               "Q3 + 1.5*IQR [K]",
               "Median A(UHII) [K]",
               "N outliers (> 1.5*IQR)",
               "N cities A(UHII) > 0",
               "N cities A(UHII) < 0",
               "N cities total"]

    multi_cols = pd.MultiIndex.from_product([list(conditions.keys()), list(kg_groups.keys())],
                                            names=["Scenario / Condition", "Climate Group"])

    df_out = pd.DataFrame(index=metrics, columns=multi_cols)
    df_out.index.name = "Metric"

    for cond_label, var_name in conditions.items():
        if var_name not in ds.data_vars:
            print(f"Warning: '{var_name}' not found, skipping")
            continue

        all_vals = ds[var_name].values

        for group_label, kg_filter in kg_groups.items():

            vals = all_vals if kg_filter is None else all_vals[kg_values == kg_filter]
            vals = vals[~np.isnan(vals)]

            q1, q3 = np.nanpercentile(vals, [25, 75])
            iqr = q3 - q1
            low_iqr = q1 - 1.5 * iqr
            high_iqr = q3 + 1.5 * iqr
            n_out = int(np.sum((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)))

            df_out.loc["Data range min [K]", (cond_label, group_label)] = f"{np.min(vals):.3f}"
            df_out.loc["Data range max [K]", (cond_label, group_label)] = f"{np.max(vals):.3f}"
            df_out.loc["Q1 - 1.5*IQR [K]", (cond_label, group_label)] = f"{low_iqr:.3f}"
            df_out.loc["Q3 + 1.5*IQR [K]", (cond_label, group_label)] = f"{high_iqr:.3f}"
            df_out.loc["Median A(UHII) [K]", (cond_label, group_label)] = f"{np.median(vals):.3f}"
            df_out.loc["N outliers (> 1.5*IQR)", (cond_label, group_label)] = n_out
            df_out.loc["N cities A(UHII) > 0", (cond_label, group_label)] = int(np.sum(vals > 0))
            df_out.loc["N cities A(UHII) < 0", (cond_label, group_label)] = int(np.sum(vals < 0))
            df_out.loc["N cities total", (cond_label, group_label)] = len(vals)

    if save_path is not None:
        import os
        df_out.to_csv(os.path.join(save_path, output_name))
        print(f"Saved to {os.path.join(save_path, output_name)}")

    return df_out

# Get data


## Complete dataset with all variables (city aggregated mean/max/min temperatures, UHII, heatwave mask, percentile thresholds etc.)

In [6]:
uhii_hist_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_hist_urb_all_var_pooled15.nc")
uhii_cont_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_cont_urb_all_var_pooled15.nc")
uhii_t2k_urb_all_var = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/UHII/UHII_per_city_extended/uhii_t2k_urb_all_var_pooled15.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


In [10]:
uhii_hist_urb_all_var

<xarray.Dataset> Size: 220MB
Dimensions:                                        (time: 3405, city: 210)
Coordinates:
  * time                                           (time) datetime64[ns] 27kB ...
  * city                                           (city) <U19 16kB 'Abu Dhab...
    location                                       (city) <U2 2kB ...
    kg_class_main                                  (city) <U1 840B ...
    kg_class_detailed                              (city) <U3 3kB ...
    kg_confidence_city                             (city) float32 840B ...
    lat                                            (city) float64 2kB ...
    lon                                            (city) float64 2kB ...
    dayofyear                                      (time) int64 27kB ...
Data variables: (12/51)
    T2M_mean_mean_urban_high                       (time, city) float64 6MB ...
    T2M_mean_max_urban_high                        (time, city) float64 6MB ...
    T2M_mean_min_urban_high                        (time, city) float64 6MB ...
    T2M_mean_mean_urban_low                        (time, city) float64 6MB ...
    T2M_mean_max_urban_low                         (time, city) float64 6MB ...
    T2M_mean_min_urban_low                         (time, city) float64 6MB ...
    ...                                             ...
    T2M_mean_mean_urban_high_extreme_p90_heatwave  (time, city) int8 715kB ...
    T2M_mean_mean_urban_high_extreme_p95_heatwave  (time, city) int8 715kB ...
    T2M_mean_mean_urban_high_extreme_p98_heatwave  (time, city) int8 715kB ...
    T2M_mean_mean_rural_extreme_p90_heatwave       (time, city) int8 715kB ...
    T2M_mean_mean_rural_extreme_p95_heatwave       (time, city) int8 715kB ...
    T2M_mean_mean_rural_extreme_p98_heatwave       (time, city) int8 715kB ...

# Calculate comprehensive summer climate metrics for multiple scenarios 
- For each city:
    - mean summer temperature
    - mean summer temperature in heatwave days 
    - UHII
    - UHII in heatwave days
- thermodynamic forcing (HIST - CONT / T2K - HIST) for all of them

## Absolute scenario differences

In [34]:
summer_climate_metrics_HIST, \
    summer_climate_metrics_CONT, \
    summer_climate_metrics_T2K, \
    summer_climate_metrics_HIST_CONT_diff, \
    summer_climate_metrics_HIST_T2K_diff = calculate_summer_climate_metrics(
        ds_hist=uhii_hist_urb_all_var,
        ds_cont=uhii_cont_urb_all_var,
        ds_t2k=uhii_t2k_urb_all_var,
        heatwave_percentiles=["p90", "p95", "p98"],
        save_path=save_data_to,
        output_name="urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/abs/summer_climate_metrics.nc")

Calculating summer climate metrics per city...

 Processing HIST scenario

Processing HIST...


/tmp/ipykernel_1063523/4272786425.py:124: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds_result = xr.concat(scenario_results, dim='city')



 Processing CONT scenario
Using HIST heatwave days for masking

Processing CONT...


/tmp/ipykernel_1063523/4272786425.py:124: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds_result = xr.concat(scenario_results, dim='city')



 Calculating HIST - CONT absolute differences

 Processing T2K scenario
Using HIST heatwave days for masking

Processing T2K...


/tmp/ipykernel_1063523/4272786425.py:124: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds_result = xr.concat(scenario_results, dim='city')



 Calculating HIST - T2K absolute differences


## Load saved data

In [3]:
summer_climate_metrics_HIST = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST.nc")
summer_climate_metrics_CONT = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_CONT.nc")
summer_climate_metrics_T2K = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_T2K.nc")
summer_climate_metrics_HIST_CONT_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST_CONT_diff.nc")
summer_climate_metrics_HIST_T2K_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST_T2K_diff.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


In [4]:
summer_climate_metrics_HIST_T2K_diff

<xarray.Dataset> Size: 91kB
Dimensions:                                                           (city: 210)
Coordinates:
  * city                                                              (city) <U19 16kB ...
    location                                                          (city) <U2 2kB ...
    kg_class_main                                                     (city) <U1 840B ...
    kg_class_detailed                                                 (city) <U3 3kB ...
    kg_confidence_city                                                (city) float32 840B ...
    lat                                                               (city) float64 2kB ...
    lon                                                               (city) float64 2kB ...
Data variables: (12/39)
    T2M_mean_mean_urban_high_summer_T2K_HIST_abs_diff                 (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_rural_summer_p98_T2K_HIST_abs_diff         (city) float64 2kB ...
    T2M_mean_mean_rural_summer_p90_T2K_HIST_abs_diff                  (city) float64 2kB ...
    T2M_mean_mean_urban_high_summer_no_hw_T2K_HIST_abs_diff           (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_rural_summer_p95_T2K_HIST_abs_diff         (city) float64 2kB ...
    T2M_mean_mean_urban_low_summer_p90_T2K_HIST_abs_diff              (city) float64 2kB ...
    ...                                                                ...
    T2M_mean_mean_urban_low_summer_p95_T2K_HIST_abs_diff              (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_urban_high_summer_p95_T2K_HIST_abs_diff    (city) float64 2kB ...
    T2M_mean_mean_rural_summer_p95_T2K_HIST_abs_diff                  (city) float64 2kB ...
    T2M_mean_mean_urban_high_summer_p95_T2K_HIST_abs_diff             (city) float64 2kB ...
    UHII_summer_p98_T2K_HIST_abs_diff                                 (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_urban_low_summer_p90_T2K_HIST_abs_diff     (city) float64 2kB ...
Attributes:
    description:           Climate scenario absolute differences (T2K - HIST)
    scenario_calculation:  T2K - HIST
    n_variables:           39
    summer_definition:     NH: JJA (Jun-Aug), SH: DJF (Dec-Feb)

# Create data table with different metrics of heatwave condition and climate class

## UHII

In [13]:
summer_metrics_per_CC_HIST = create_uhii_summary_table(summer_climate_metrics_HIST,
                                                      save_path=save_tables_to,
                                                      output_name="Summer_metrics_UHII_HIST_CC.csv")
summer_metrics_per_CC_HIST

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_UHII_HIST_CC.csv


Scenario / Condition   HIST All Summer Days                             \
Climate Group                    All cities Tropical    Arid Temperate   
Metric                                                                   
Data range min [K]                   -3.881   -1.435  -2.416    -3.881   
Data range max [K]                    9.419    6.438   4.939     5.612   
Q1 - 1.5*IQR [K]                     -1.201   -1.978  -2.472    -0.936   
Q3 + 1.5*IQR [K]                      2.738    3.361   4.574     2.502   
Median UHII [K]                       0.556    0.296   0.852     0.618   
N outliers (> 1.5*IQR)                   25        2       1        10   
N cities UHII > 0                       185       33      25        78   
N cities UHII < 0                        25        9       6         8   
N cities total                          210       42      31        86   

Scenario / Condition               HIST No-HW Summer Days                   \
Climate Group          Continental             All cities Tropical    Arid   
Metric                                                                       
Data range min [K]          -0.058                 -3.873   -1.480  -2.414   
Data range max [K]           9.419                  9.386    6.455   4.846   
Q1 - 1.5*IQR [K]            -0.486                 -1.224   -1.996  -2.482   
Q3 + 1.5*IQR [K]             1.704                  2.747    3.377   4.576   
Median UHII [K]              0.555                  0.553    0.293   0.842   
N outliers (> 1.5*IQR)           6                     25        2       1   
N cities UHII > 0               49                    185       32      25   
N cities UHII < 0                2                     25       10       6   
N cities total                  51                    210       42      31   

Scenario / Condition                         HIST p95-HW Summer Days           \
Climate Group          Temperate Continental              All cities Tropical   
Metric                                                                          
Data range min [K]        -3.873      -0.078                  -3.406   -0.747   
Data range max [K]         5.617       9.386                  10.174    6.232   
Q1 - 1.5*IQR [K]          -0.993      -0.482                  -1.495   -1.530   
Q3 + 1.5*IQR [K]           2.545       1.689                   3.388    2.884   
Median UHII [K]            0.608       0.552                   0.755    0.313   
N outliers (> 1.5*IQR)        10           6                      19        5   
N cities UHII > 0             78          50                     190       37   
N cities UHII < 0              8           1                      19        5   
N cities total                86          51                     209       42   

Scenario / Condition                                  
Climate Group             Arid Temperate Continental  
Metric                                                
Data range min [K]      -2.794    -3.406      -0.242  
Data range max [K]       7.304     5.846      10.174  
Q1 - 1.5*IQR [K]        -2.110    -1.218      -0.464  
Q3 + 1.5*IQR [K]         4.333     3.461       1.989  
Median UHII [K]          1.087     0.949       0.690  
N outliers (> 1.5*IQR)       2         6           6  
N cities UHII > 0           26        78          49  
N cities UHII < 0            5         7           2  
N cities total              31        85          51

## $\Delta$UHII

### HIST - CONT

In [14]:
summer_metrics_per_CC_HIST_CONT_diff = create_delta_uhii_summary_table(summer_climate_metrics_HIST_CONT_diff, 
                                                                      save_path=save_tables_to,
                                                                      output_name="Summer_metrics_Delta_UHII_HIST-CONT_CC.csv")
summer_metrics_per_CC_HIST_CONT_diff

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_Delta_UHII_HIST-CONT_CC.csv


Scenario / Condition      HIST-CONT All Summer Days                   \
Climate Group                            All cities Tropical    Arid   
Metric                                                                 
Data range min [K]                           -0.376   -0.049  -0.184   
Data range max [K]                            0.366    0.052   0.139   
Q1 - 1.5*IQR [K]                             -0.079   -0.069  -0.123   
Q3 + 1.5*IQR [K]                              0.073    0.061   0.102   
Median $\Delta$UHII [K]                      -0.002   -0.004  -0.004   
N outliers (> 1.5*IQR)                           17        0       3   
N cities $\Delta$UHII > 0                       100       20      12   
N cities $\Delta$UHII < 0                       110       22      19   
N cities total                                  210       42      31   

Scenario / Condition                            HIST-CONT No-HW Summer Days  \
Climate Group             Temperate Continental                  All cities   
Metric                                                                        
Data range min [K]           -0.376      -0.139                      -0.357   
Data range max [K]            0.366       0.076                       0.365   
Q1 - 1.5*IQR [K]             -0.074      -0.055                      -0.080   
Q3 + 1.5*IQR [K]              0.070       0.060                       0.074   
Median $\Delta$UHII [K]      -0.003       0.003                      -0.000   
N outliers (> 1.5*IQR)            7           5                          17   
N cities $\Delta$UHII > 0        39          29                         102   
N cities $\Delta$UHII < 0        47          22                         108   
N cities total                   86          51                         210   

Scenario / Condition                                              \
Climate Group             Tropical    Arid Temperate Continental   
Metric                                                             
Data range min [K]          -0.050  -0.187    -0.357      -0.140   
Data range max [K]           0.053   0.134     0.365       0.080   
Q1 - 1.5*IQR [K]            -0.075  -0.122    -0.078      -0.060   
Q3 + 1.5*IQR [K]             0.071   0.101     0.071       0.063   
Median $\Delta$UHII [K]     -0.001  -0.004    -0.003       0.001   
N outliers (> 1.5*IQR)           0       4         7           5   
N cities $\Delta$UHII > 0       20      13        41          28   
N cities $\Delta$UHII < 0       22      18        45          23   
N cities total                  42      31        86          51   

Scenario / Condition      HIST-CONT p95-HW Summer Days                   \
Climate Group                               All cities Tropical    Arid   
Metric                                                                    
Data range min [K]                              -0.758   -0.097  -0.130   
Data range max [K]                               0.418    0.139   0.301   
Q1 - 1.5*IQR [K]                                -0.166   -0.134  -0.176   
Q3 + 1.5*IQR [K]                                 0.201    0.121   0.229   
Median $\Delta$UHII [K]                          0.008   -0.002  -0.002   
N outliers (> 1.5*IQR)                              18        1       3   
N cities $\Delta$UHII > 0                          112       20      15   
N cities $\Delta$UHII < 0                           97       22      16   
N cities total                                     209       42      31   

Scenario / Condition                             
Climate Group             Temperate Continental  
Metric                                           
Data range min [K]           -0.758      -0.197  
Data range max [K]            0.418       0.296  
Q1 - 1.5*IQR [K]             -0.185      -0.142  
Q3 + 1.5*IQR [K]              0.232       0.194  
Median $\Delta$UHII [K]       0.019       0.022  
N outliers (> 1.5*IQR)            7           6  
N cities $\Delta$U

### T2K - HIST

In [12]:
summer_metrics_per_CC_T2K_HIST_diff = create_delta_uhii_summary_table(summer_climate_metrics_HIST_T2K_diff,
                                                                     save_path=save_tables_to,
                                                                     output_name="Summer_metrics_Delta_UHII_T2K-HIST_CC.csv")
summer_metrics_per_CC_T2K_HIST_diff

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_Delta_UHII_T2K-HIST_CC.csv


Scenario / Condition      T2K-HIST All Summer Days                             \
Climate Group                           All cities Tropical    Arid Temperate   
Metric                                                                          
Data range min [K]                          -0.393   -0.079  -0.185    -0.393   
Data range max [K]                           0.183    0.045   0.124     0.106   
Q1 - 1.5*IQR [K]                            -0.070   -0.067  -0.114    -0.064   
Q3 + 1.5*IQR [K]                             0.072    0.053   0.089     0.068   
Median $\Delta$UHII [K]                      0.003   -0.009  -0.001     0.005   
N outliers (> 1.5*IQR)                          22        1       5         6   
N cities $\Delta$UHII > 0                      114       14      15        53   
N cities $\Delta$UHII < 0                       96       28      16        33   
N cities total                                 210       42      31        86   

Scenario / Condition                  T2K-HIST No-HW Summer Days           \
Climate Group             Continental                 All cities Tropical   
Metric                                                                      
Data range min [K]             -0.160                     -0.364   -0.072   
Data range max [K]              0.183                      0.189    0.043   
Q1 - 1.5*IQR [K]               -0.059                     -0.068   -0.067   
Q3 + 1.5*IQR [K]                0.074                      0.069    0.057   
Median $\Delta$UHII [K]         0.004                      0.001   -0.009   
N outliers (> 1.5*IQR)              8                         23        1   
N cities $\Delta$UHII > 0          32                        110       15   
N cities $\Delta$UHII < 0          19                        100       27   
N cities total                     51                        210       42   

Scenario / Condition                                     \
Climate Group                Arid Temperate Continental   
Metric                                                    
Data range min [K]         -0.181    -0.364      -0.159   
Data range max [K]          0.101     0.106       0.189   
Q1 - 1.5*IQR [K]           -0.111    -0.056      -0.058   
Q3 + 1.5*IQR [K]            0.085     0.061       0.073   
Median $\Delta$UHII [K]    -0.003     0.004       0.002   
N outliers (> 1.5*IQR)          5         5           8   
N cities $\Delta$UHII > 0      15        50          30   
N cities $\Delta$UHII < 0      16        36          21   
N cities total                 31        86          51   

Scenario / Condition      T2K-HIST p95-HW Summer Days                   \
Climate Group                              All cities Tropical    Arid   
Metric                                                                   
Data range min [K]                             -1.093   -0.146  -0.308   
Data range max [K]                              0.490    0.178   0.455   
Q1 - 1.5*IQR [K]                               -0.196   -0.111  -0.307   
Q3 + 1.5*IQR [K]                                0.205    0.084   0.351   
Median $\Delta$UHII [K]                         0.005   -0.019  -0.018   
N outliers (> 1.5*IQR)                             23        3       2   
N cities $\Delta$UHII > 0                         109       14      11   
N cities $\Delta$UHII < 0                         100       28      20   
N cities total                                    209       42      31   

Scenario / Condition                             
Climate Group             Temperate Continental  
Metric                                           
Data range min [K]           -1.093      -0.494  
Data range max [K]            0.490       0.246  
Q1 - 1.5*IQR [K]             -0.233      -0.097  
Q3 + 1.5*IQR [K]              0.231       0.164  
Median $\Delta$UHII [K]       0.006       0.036  
N outliers (> 1.5*IQR)           10          10  
N cities $\Delta$UHII > 0        46          38  
N ci

# Amplification Index (heatwave vs. non-heatwave)
Here, I will calculate the <b>amplitfication index <i>A</i></b> in order to define the difference in UHII, T_urban and T_rural between heatwave conditions and non-heatwave conditions. 
Herefor I am using our `summer_climate_metrics.nc`files for each scenario. 

$A_UHII= UHII(hw p95) - UHII(no-hw)$


## Absolute summer mean metrics

In [5]:
summer_climate_metrics_HIST = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST.nc")
summer_climate_metrics_CONT = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_CONT.nc")
summer_climate_metrics_T2K = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_T2K.nc")
summer_climate_metrics_HIST_CONT_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST_CONT_diff.nc")
summer_climate_metrics_HIST_T2K_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/summer_climate_metrics_HIST_T2K_diff.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


## Calculate amplification index <i>A</i>
For `T2M_mean_mean_urban_high_summer`, `T2M_mean_mean_rural_summer`, `UHII_summer` between no-heatwave days and p95 heatwave days.

In [9]:
amplification_HIST = calculate_amplification_index(summer_climate_metrics_HIST)

In [10]:
amplification_HIST

<xarray.Dataset> Size: 96kB
Dimensions:                                         (city: 210)
Coordinates:
  * city                                            (city) <U19 16kB 'Abu Dha...
    location                                        (city) <U2 2kB ...
    kg_class_main                                   (city) <U1 840B ...
    kg_class_detailed                               (city) <U3 3kB ...
    kg_confidence_city                              (city) float32 840B ...
    lat                                             (city) float64 2kB ...
    lon                                             (city) float64 2kB ...
Data variables: (12/42)
    T2M_mean_mean_urban_high_summer                 (city) float64 2kB ...
    T2M_mean_mean_urban_low_summer                  (city) float64 2kB ...
    T2M_mean_mean_rural_summer                      (city) float64 2kB ...
    UHII_summer                                     (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_urban_high_summer        (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_urban_low_summer         (city) float64 2kB ...
    ...                                              ...
    Diurnal_Tmax_Tmin_diff_urban_high_summer_p98    (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_urban_low_summer_p98     (city) float64 2kB ...
    Diurnal_Tmax_Tmin_diff_rural_summer_p98         (city) float64 2kB ...
    amplification_T2M_mean_mean_urban_high_summer   (city) float64 2kB 3.188 ...
    amplification_T2M_mean_mean_rural_summer        (city) float64 2kB 3.096 ...
    amplification_UHII_summer                       (city) float64 2kB 0.0920...
Attributes:
    description:         Summer climate metrics for HIST scenario
    summer_definition:   NH: JJA (Jun-Aug), SH: DJF (Dec-Feb)
    scenario:            HIST
    heatwave_reference:  self
    note:                Heatwave days defined by HIST dataset for all scenarios

In [11]:
amplification_CONT = calculate_amplification_index(summer_climate_metrics_CONT)
amplification_T2K = calculate_amplification_index(summer_climate_metrics_T2K)

In [12]:
amplification_HIST_CONT_diff = calculate_amplification_index(summer_climate_metrics_HIST_CONT_diff, suffix="HIST_CONT_abs_diff")
amplification_T2K_HIST_diff = calculate_amplification_index(summer_climate_metrics_HIST_T2K_diff, suffix="T2K_HIST_abs_diff")

### Save data

In [28]:
amplification_HIST.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_HIST.nc")
amplification_CONT.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_CONT.nc")
amplification_T2K.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_T2K.nc")
amplification_HIST_CONT_diff.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_HIST_CONT_diff.nc")
amplification_T2K_HIST_diff.to_netcdf(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_T2K_HIST_diff.nc")

### Load saved data

In [6]:
amplification_HIST = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_HIST.nc")
amplification_CONT = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_CONT.nc")
amplification_T2K = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_T2K.nc")
amplification_HIST_CONT_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_HIST_CONT_diff.nc")
amplification_T2K_HIST_diff = xr.open_dataset(save_data_to + "urban/ensemble_mean/include_city_class/Summer_metrics_thermodynamics/absolute_scenario_diff/amplification_T2K_HIST_diff.nc")

sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found
sh: getfattr: command not found


## Create datatables with A($\Delta$UHII) for HIST - CONT and T2K - HIST

### HIST

In [35]:
summer_metrics_per_CC_A_UHII_HIST = create_a_uhii_summary_table(amplification_HIST,
                                                      save_path=save_tables_to,
                                                      output_name="Summer_metrics_A_UHII_HIST_CC.csv")
summer_metrics_per_CC_A_UHII_HIST

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_A_UHII_HIST_CC.csv


Scenario / Condition         HIST                                       
Climate Group          All cities Tropical    Arid Temperate Continental
Metric                                                                  
Data range min [K]         -1.139   -0.409  -0.832    -1.139      -0.425
Data range max [K]          2.458    1.054   2.458     1.586       1.920
Q1 - 1.5*IQR [K]           -0.676   -0.364  -0.659    -0.568      -0.614
Q3 + 1.5*IQR [K]            1.119    0.480   0.853     1.219       1.056
Median A(UHII) [K]          0.179    0.039   0.108     0.374       0.190
N outliers (> 1.5*IQR)         10        4       2         7           1
N cities A(UHII) > 0          156       26      21        71          38
N cities A(UHII) < 0           53       16      10        14          13
N cities total                209       42      31        85          51

### HIST - CONT

In [43]:
summer_metrics_per_CC_A_UHII_HIST_CONT_diff = create_a_uhii_summary_table(amplification_HIST_CONT_diff,
                                                      save_path=save_tables_to,
                                                      output_name="Summer_metrics_A_UHII_HIST_CONT_CC.csv")
summer_metrics_per_CC_A_UHII_HIST_CONT_diff

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_A_UHII_HIST_CONT_CC.csv


Scenario / Condition   HIST - CONT                                       
Climate Group           All cities Tropical    Arid Temperate Continental
Metric                                                                   
Data range min [K]          -0.798   -0.076  -0.130    -0.798      -0.167
Data range max [K]           0.486    0.123   0.486     0.395       0.291
Q1 - 1.5*IQR [K]            -0.168   -0.094  -0.226    -0.182      -0.169
Q3 + 1.5*IQR [K]             0.203    0.093   0.290     0.221       0.223
Median A(UHII) [K]           0.010    0.001   0.006     0.026       0.013
N outliers (> 1.5*IQR)          16        1       2        10           3
N cities A(UHII) > 0           121       22      17        54          28
N cities A(UHII) < 0            88       20      14        31          23
N cities total                 209       42      31        85          51

### T2K - HIST

In [46]:
summer_metrics_per_CC_A_UHII_T2K_HIST_diff = create_a_uhii_summary_table(amplification_T2K_HIST_diff,
                                                      save_path=save_tables_to,
                                                      output_name="Summer_metrics_A_UHII_T2k_HIST_CC.csv")
summer_metrics_per_CC_A_UHII_T2K_HIST_diff

Saved to /home/b/b383801/Final_Notebooks/New_data/ensemble_mean/Tables/Summer_metrics_A_UHII_T2k_HIST_CC.csv


Scenario / Condition   HIST - CONT                                       
Climate Group           All cities Tropical    Arid Temperate Continental
Metric                                                                   
Data range min [K]          -0.729   -0.076  -0.268    -0.729      -0.456
Data range max [K]           0.437    0.135   0.424     0.437       0.232
Q1 - 1.5*IQR [K]            -0.198   -0.100  -0.285    -0.246      -0.142
Q3 + 1.5*IQR [K]             0.199    0.088   0.344     0.229       0.172
Median A(UHII) [K]           0.007   -0.009   0.007     0.009       0.017
N outliers (> 1.5*IQR)          17        2       2         9           3
N cities A(UHII) > 0           113       18      16        46          33
N cities A(UHII) < 0            96       24      15        39          18
N cities total                 209       42      31        85          51